# CRF で固有表現抽出器を学習する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/10_sequence_labeling/crf_tagger.ipynb)

NLP クックブック 10-04 ／ 系列ラベリング

ニューラルネットを使わずに、手作りの特徴量と条件付き確率場 (CRF) で系列ラベリングをしたい。CPU で数十秒で学習できる。

In [ ]:
!pip install -q sklearn-crfsuite seqeval

In [ ]:
import nltk
nltk.download("conll2002", quiet=True)
from nltk.corpus import conll2002

train = list(conll2002.iob_sents("esp.train"))   # スペイン語の固有表現データ: (単語, 品詞, BIO タグ)
test = list(conll2002.iob_sents("esp.testb"))
print(len(train), len(test), train[0][:6])

def word_features(sent, i):
    w, pos = sent[i][0], sent[i][1]
    f = {"bias": 1.0, "lower": w.lower(), "suffix3": w[-3:], "suffix2": w[-2:], "is_upper": w.isupper(),
         "is_title": w.istitle(), "is_digit": w.isdigit(), "pos": pos}
    for d in (-1, 1):   # 前後 1 語の特徴
        if 0 <= i + d < len(sent):
            w2, pos2 = sent[i + d][0], sent[i + d][1]
            f.update({f"{d}:lower": w2.lower(), f"{d}:is_title": w2.istitle(), f"{d}:pos": pos2})
        else:
            f["BOS" if d == -1 else "EOS"] = True
    return f

X_train = [[word_features(s, i) for i in range(len(s))] for s in train]
y_train = [[t for _, _, t in s] for s in train]
X_test = [[word_features(s, i) for i in range(len(s))] for s in test]
y_test = [[t for _, _, t in s] for s in test]

In [ ]:
import sklearn_crfsuite
from seqeval.metrics import classification_report

crf = sklearn_crfsuite.CRF(algorithm="lbfgs", c1=0.1, c2=0.1, max_iterations=100, all_possible_transitions=True)
crf.fit(X_train, y_train)
print(classification_report(y_test, crf.predict(X_test), digits=3))

In [ ]:
# 学習された遷移の重み: O → I-xxx のような不正な遷移には大きな負の重みが付く
from collections import Counter

trans = Counter(crf.transition_features_)
print("起きやすい遷移:", [(a, b, round(w, 1)) for (a, b), w in trans.most_common(5)])
print("起きにくい遷移:", [(a, b, round(w, 1)) for (a, b), w in trans.most_common()[-5:]])

- CRF はラベル間の遷移も学習するので、B- なしの I- のような不正な系列を出しにくい。
- 特徴量を自分で設計できる・学習が速い・挙動を解釈できるのが利点。少量データや計算資源が限られる場面では今でも有力。